<a href="https://colab.research.google.com/github/fabio-la-torre/smartphone-trade-in-price-estimator/blob/main/trade_in_price_estimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [126]:
import pandas as pd
import numpy as np

# Load the dataset directly from this GitHub repository
url = "https://raw.githubusercontent.com/fabio-la-torre/smartphone-trade-in-price-estimator/main/used_device_data.csv"
df = pd.read_csv(url)
total = len(df)

df = df[df["screen_size"] <= 17.8]  # keep only phones (screen up to 7 inches)

print("Devices in dataset:", total)
print("Tablets removed:", total - len(df))
print("Phones left:", len(df))
df.head()

Devices in dataset: 3454
Tablets removed: 275
Phones left: 3179


,device_brand,os,screen_size,4g,5g,rear_camera_mp,front_camera_mp,internal_memory,ram,battery,weight,release_year,days_used,normalized_used_price,normalized_new_price
0,Honor,Android,14.50,yes,no,13.0,5.0,64.0,3.0,3020.0,146.0,2020,127,4.307572,4.715100
1,Honor,Android,17.30,yes,yes,13.0,16.0,128.0,8.0,4300.0,213.0,2020,325,5.162097,5.519018
2,Honor,Android,16.69,yes,yes,13.0,8.0,128.0,8.0,4200.0,213.0,2020,162,5.111084,5.884631
4,Honor,Android,15.32,yes,no,13.0,8.0,64.0,3.0,5000.0,185.0,2020,293,4.389995,4.947837
5,Honor,Android,16.23,yes,no,13.0,8.0,64.0,4.0,4000.0,176.0,2020,223,4.413889,5.060694


In [127]:
df.corr(numeric_only=True)["normalized_used_price"].sort_values()

,normalized_used_price
days_used,-0.379221
internal_memory,0.166357
weight,0.424992
ram,0.538020
release_year,0.543893
rear_camera_mp,0.655542
front_camera_mp,0.659446
battery,0.665767
screen_size,0.685202
normalized_new_price,0.832182


In [128]:
# Feature selection experiments (phones only)
# Each experiment trains a model with a different set of features and prints its R² score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df["5g"] = df["5g"] == "yes"   # convert yes/no to True/False
df["4g"] = df["4g"] == "yes"

def test_features(feature_list):
    d = df[feature_list + ["normalized_used_price"]].dropna()
    X_tr, X_te, y_tr, y_te = train_test_split(d[feature_list], d["normalized_used_price"],
                                              test_size=0.2, random_state=42)
    m = LinearRegression()
    m.fit(X_tr, y_tr)
    return round(m.score(X_te, y_te), 3)

base = ["normalized_new_price", "release_year", "days_used", "ram"]

experiments = {
    "1. Base + battery + 5G":                 base + ["battery", "5g"],
    "2. Internal memory instead of battery":  base + ["internal_memory", "5g"],
    "3. Add cameras":                         base + ["battery", "5g", "rear_camera_mp", "front_camera_mp"],
    "4. 4G instead of 5G":                    base + ["battery", "4g", "rear_camera_mp", "front_camera_mp"],
    "5. No 4G/5G (simpler, same score)":      base + ["battery", "rear_camera_mp", "front_camera_mp"],
    "6. Add screen size (final model)":       base + ["battery", "screen_size", "rear_camera_mp", "front_camera_mp"],
}

for name, feature_list in experiments.items():
    print(test_features(feature_list), "-", name)

0.786 - 1. Base + battery + 5G
0.779 - 2. Internal memory instead of battery
0.804 - 3. Add cameras
0.803 - 4. 4G instead of 5G
0.803 - 5. No 4G/5G (simpler, same score)
0.82 - 6. Add screen size (final model)


In [129]:
# Train and evaluate the model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Features chosen after the experiments (see previous cell)
features = ["normalized_new_price", "release_year", "days_used", "ram",
            "battery", "screen_size", "rear_camera_mp", "front_camera_mp"]

# Keep only phones that have a value for every feature
data = df[features + ["normalized_used_price"]].dropna()
print("Phones with complete data:", len(data))

X = data[features]                   # inputs: phone characteristics
y = data["normalized_used_price"]    # target: used price (log scale)

# Split the phones: 80% to train the model, 20% to test it on phones it has never seen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set:", len(X_train), "phones (80%)")
print("Test set:", len(X_test), "phones (20%)")

# Training: the model learns one weight for each feature
model = LinearRegression()
model.fit(X_train, y_train)

# R² score on the test set: share of price differences the model explains (1.0 = perfect)
score = model.score(X_test, y_test)
print("R² score on test set:", round(score, 3))

Phones with complete data: 2991
Training set: 2392 phones (80%)
Test set: 599 phones (20%)
R² score on test set: 0.82


In [130]:
# Weights learned by the model (effect of +1 unit of each feature on the log price)
for name, weight in zip(features, model.coef_):
    print(f"{name}: {weight:.6f}")

normalized_new_price: 0.438142
release_year: 0.027093
days_used: 0.000061
ram: 0.014307
battery: 0.000023
screen_size: 0.042648
rear_camera_mp: 0.018973
front_camera_mp: 0.011464


In [131]:
# Estimate the trade-in value of a phone in euros
def estimate_price(new_price_eur, release_year, days_used, ram, battery_mah,
                   screen_inches, rear_mp, front_mp):
    phone = pd.DataFrame([{
        "normalized_new_price": np.log(new_price_eur),  # euros -> log scale used by the model
        "release_year": release_year,
        "days_used": days_used,
        "ram": ram,                                     # GB
        "battery": battery_mah,                         # mAh
        "screen_size": screen_inches * 2.54,            # inches -> cm
        "rear_camera_mp": rear_mp,
        "front_camera_mp": front_mp,
    }])
    log_price = model.predict(phone)[0]
    return round(np.exp(log_price))                     # log scale -> euros

# Example: 300 € phone, released 2020, used 300 days, 4 GB RAM, 4000 mAh, 6.2", 10 MP + 10 MP
print(estimate_price(300, 2020, 300, 4, 4000, 6.2, 10, 10), "€")

124 €
